In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights



In [2]:

# 1. Custom Dataset to load and label /images dynamically
class LocalImageDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Scan directory and label based on file extension
        if os.path.exists(image_dir):
            for filename in os.listdir(image_dir):
                if len(self.image_paths) >= 5000:
                    break

                filepath = os.path.join(image_dir, filename)
                ext = filename.lower()

                if ext.endswith('.png'):
                    self.image_paths.append(filepath)
                    self.labels.append(0)  # PNG = 0
                elif ext.endswith('.jpg') or ext.endswith('.jpeg'):
                    self.image_paths.append(filepath)
                    self.labels.append(1)  # JPG/JPEG = 1
        else:
            print(f"Warning: Directory {image_dir} not found.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Open image and force RGB mode (handles grayscale or 4-channel RGBA consistency)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label



In [3]:
# 2. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Data transforms tailored for ResNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Explicitly resize all varying image dimensions
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset from /images
dataset = LocalImageDataset(image_dir='E:\images', transform=transform)

# Safe check to prevent division-by-zero errors if the folder is empty
if len(dataset) == 0:
    raise ValueError("No matching .jpg or .png images found in /images directory.")

train_loader = DataLoader(dataset, batch_size=1024, shuffle=True, num_workers=0)

# 4. Load pretrained ResNet-18 and modify the final layer for 2 classes (PNG vs JPG)
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification setup
model = model.to(device)

# 5. Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 6. Training loop
num_epochs = 10
model.train()



Using device: cuda


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
for epoch in range(num_epochs):
    print('a')
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        print('b')
        images, labels = images.to(device), labels.to(device)
        print('c')
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        print('d')
        # Backward and optimize
        optimizer.zero_grad()
        print('e')
        loss.backward()
        print('f')
        optimizer.step()

        print('g')
        running_loss += loss.item()
        if (i + 1) % max(1, len(train_loader) // 5) == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/max(1, i+1):.4f}")

print("Finished Training")


a
b
c
d
e
f
g
Epoch [1/10], Step [1/5], Loss: 0.6737
b
c
d
e
f
g
Epoch [1/10], Step [2/5], Loss: 0.6553
b
c
d
e
f
g
Epoch [1/10], Step [3/5], Loss: 0.6422
b
c
d
e
f
g
Epoch [1/10], Step [4/5], Loss: 0.6376
b
c
d
e
f
g
Epoch [1/10], Step [5/5], Loss: 0.6286
a
b
c
d
e
f
g
Epoch [2/10], Step [1/5], Loss: 0.5808
b
c
d
e
f
g
Epoch [2/10], Step [2/5], Loss: 0.5664
b
c
d
e
f
g
Epoch [2/10], Step [3/5], Loss: 0.5525
b
c
d
e
f
g
Epoch [2/10], Step [4/5], Loss: 0.5418
b
c
d
e
f
g
Epoch [2/10], Step [5/5], Loss: 0.5269
a
b
c
d
e
f
g
Epoch [3/10], Step [1/5], Loss: 0.4423
b
c
d
e
f
g
Epoch [3/10], Step [2/5], Loss: 0.4291
b
c
d
e
f
g
Epoch [3/10], Step [3/5], Loss: 0.4198
b
c
d
e
f
g
Epoch [3/10], Step [4/5], Loss: 0.4048
b
c
d
e
f
g
Epoch [3/10], Step [5/5], Loss: 0.3903
a
b
c
d
e
f
g
Epoch [4/10], Step [1/5], Loss: 0.3034
b
c
d
e
f
g
Epoch [4/10], Step [2/5], Loss: 0.2917
b
c
d
e
f
g
Epoch [4/10], Step [3/5], Loss: 0.2764
b
c
d
e
f
g
Epoch [4/10], Step [4/5], Loss: 0.2627
b
c
d
e
f
g
Epoch [4/10


# 7. Save model parameters as a .pth file
model_save_path = "./resnet18_image_classifier.pth"
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
torch.save(model.state_dict(), model_save_path)
print(f"Model saved successfully to {model_save_path}")

In [5]:
# 7. COMPILE AND SAVE WITH TORCHSCRIPT (Self-contained file)
model.eval()  # Must place in evaluation mode before tracing
with torch.no_file_operations_context if hasattr(torch, 'no_file_operations_context') else torch.inference_mode():
    # Create dummy input on the exact same device matching expected dimensions
    dummy_input = torch.rand(1, 3, 224, 224).to(device)
    # Trace compiles the model structure + parameter weights into an intermediate graph
    traced_model = torch.jit.trace(model, dummy_input)

    model_save_path = "./resnet18_compiled.pt"
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    traced_model.save(model_save_path)
    print(f"ResNet-18 serialized successfully to {model_save_path}")

ResNet-18 serialized successfully to ./resnet18_compiled.pt
